# MLP Dead Neuron Analysis

Identify and characterize inactive neurons: dead neurons,
activation frequency, magnitude distribution, and near-dead detection.

## Why This Matters

MLP dead neuron provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_dead_neuron_analysis import (
    dead_neuron_detection, neuron_activation_frequency,
    neuron_activation_magnitude_distribution, near_dead_neurons,
    dead_neuron_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Dead Neuron Detection

Neurons that never activate above threshold waste capacity.

In [ ]:
for layer in range(4):
    result = dead_neuron_detection(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_dead']}/{result['total_neurons']} dead "
          f"({result['dead_fraction']:.1%})")
    if result['dead_indices']:
        print(f"    Dead indices: {result['dead_indices'][:10]}")

## Activation Frequency

How often does each neuron fire across positions?

In [ ]:
for layer in range(4):
    result = neuron_activation_frequency(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_freq={result['mean_frequency']:.3f}, "
          f"always={result['n_always_active']}, never={result['n_never_active']}")
    for bin_name, count in result['histogram'].items():
        bar = '█' * (count // 2)
        print(f"    {bin_name}: {count} {bar}")

## Activation Magnitude Distribution

How are activation magnitudes distributed?

In [ ]:
for layer in range(4):
    result = neuron_activation_magnitude_distribution(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean={result['global_mean']:.4f}, "
          f"max={result['global_max']:.4f}, "
          f"std={result['mean_std']:.4f}, "
          f"high_var={result['n_high_variance']}")

## Near-Dead Neurons

Neurons with very low activation that may be nearly dead.

In [ ]:
for layer in range(4):
    result = near_dead_neurons(model, tokens, layer=layer)
    print(f"  Layer {layer}: {result['n_near_dead']} near-dead (thresh={result['threshold']:.6f})")
    for n in result['weakest_neurons'][:5]:
        print(f"    N{n['neuron']}: mean_act={n['mean_activation']:.6f}")

## Dead Neuron Summary

In [ ]:
result = dead_neuron_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: dead={p['dead_fraction']:.1%}, "
          f"freq={p['mean_frequency']:.3f}, never={p['n_never_active']}")